### MFR Demo

In [1]:
import torch

import pandas as pd
from datasets import load_dataset
from utils import counting, mfr, evaluate

import random
import numpy as np

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

g = torch.Generator()
g.manual_seed(SEED)

# Load data
dev_pub_data = load_dataset("weerayut/multilexnorm2026-dev-pub")

# Select data
train, val = dev_pub_data["train"], dev_pub_data["validation"]
#train = dev_pub_data["train"].select(range(500))
#val = dev_pub_data["validation"].select(range(100))

c:\Users\DH\Desktop\플래너\5. 학기 자료\26-1\2.인지개\과제\MultiLexNorm2026-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### Submission demo

In [2]:
import random
import numpy as np
import torch
import pandas as pd

from datasets import load_dataset
from torch.utils.data import DataLoader
from transformers import AutoTokenizer

from utils import counting, mfr, evaluate

from train_module import (
    build_label_vocab,
    TokenNormDataset,
    train_model,
    predict_with_mfr_fallback,
    evaluate_predictions,
)

In [3]:
print(type(train))
print(len(train))
print(train[0])
print(train.column_names)

print(type(val))
print(len(val))
print(val[0])
print(val.column_names)

<class 'datasets.arrow_dataset.Dataset'>
39178
{'raw': ['Dette', 'er', 'ikke', 'tilfaeldigt', '.'], 'norm': ['Dette', 'er', 'ikke', 'tilfældigt', '.'], 'lang': 'da'}
['raw', 'norm', 'lang']
<class 'datasets.arrow_dataset.Dataset'>
8408
{'raw': ['fänd', 'ich', 'toll', '!', '%PosSmiley'], 'norm': ['Fände', 'ich', 'toll', '!', '%PosSmiley'], 'lang': 'de'}
['raw', 'norm', 'lang']


In [4]:
def count_length_mismatch(dataset):
    mismatch = 0
    examples = []

    for i, item in enumerate(dataset):
        if len(item["raw"]) != len(item["norm"]):
            mismatch += 1
            if len(examples) < 5:
                examples.append((i, item["raw"], item["norm"]))

    return mismatch, examples


train_mismatch, train_examples = count_length_mismatch(train)
val_mismatch, val_examples = count_length_mismatch(val)

print("train mismatch:", train_mismatch)
print("val mismatch:", val_mismatch)

print("train mismatch examples:")
for ex in train_examples:
    print(ex)

print("val mismatch examples:")
for ex in val_examples:
    print(ex)

train mismatch: 0
val mismatch: 0
train mismatch examples:
val mismatch examples:


길이 맞는 데이터만 사용

In [5]:
# original train: 39178
# filtered train: 39178
# original val: 8408
# filtered val: 8408
# 따라서 밑 함수 안쓰는 걸로.
# def filter_same_length(dataset):
#    return dataset.filter(lambda x: len(x["raw"]) == len(x["norm"]))
train_tok = train
val_tok = val



MFR baseline 재현

In [6]:
counts = counting(train_tok)

raw_val = [x["raw"] for x in val_tok]
gold_val = [x["norm"] for x in val_tok]

mfr_pred_val = [mfr(x, counts) for x in raw_val]

lai, acc, err = evaluate(
    raw_val,
    gold_val,
    mfr_pred_val,
    ignCaps=False,
    verbose=False,
    info=True,
)

print("MFR result")
print("LAI:", lai)
print("Accuracy:", acc)
print("ERR:", err)

Baseline acc.(LAI): 88.48
Accuracy:           92.97
ERR:                39.02
MFR result
LAI: 0.8847954569019836
Accuracy: 0.9297478803927355
ERR: 0.3901966214345056


LAI baseline

In [7]:
raw_pred_val = raw_val

lai_raw, acc_raw, err_raw = evaluate(
    raw_val,
    gold_val,
    raw_pred_val,
    ignCaps=False,
    verbose=False,
    info=True,
)

print("Raw baseline result")
print("LAI:", lai_raw)
print("Accuracy:", acc_raw)
print("ERR:", err_raw)

Baseline acc.(LAI): 88.48
Accuracy:           88.48
ERR:                0.00
Raw baseline result
LAI: 0.8847954569019836
Accuracy: 0.8847954569019836
ERR: 0.0


Model / tokenizer 설정

In [ ]:
MODEL_NAME = "distilbert-base-multilingual-cased"
MAX_LEN = 128
BATCH_SIZE = 16
EPOCHS = 3
LR = 2e-5

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

c:\Users\DH\Desktop\플래너\5. 학기 자료\26-1\2.인지개\과제\MultiLexNorm2026-main\.venv\Lib\site-packages\huggingface_hub\file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


label vocab 생성
-전체 언어를 하나로 학습하므로 train 전체의 norm token을 label vocabulary로 만듦.

In [9]:
label2id, id2label = build_label_vocab(
    train_tok,
    min_freq=1,
    add_unk=True,
)

num_labels = len(label2id)

print("num_labels:", num_labels)

for i, item in enumerate(list(label2id.items())[:20]):
    print(i, item)

num_labels: 115445
0 ('<UNK>', 0)
1 ('_', 1)
2 ('.', 2)
3 (',', 3)
4 ('i', 4)
5 ('da', 5)
6 ('je', 6)
7 ('。', 7)
8 ('ไม่', 8)
9 ('không', 9)
10 ('の', 10)
11 (':', 11)
12 ('?', 12)
13 ('ที่', 13)
14 ('が', 14)
15 ('se', 15)
16 ('!', 16)
17 ('"', 17)
18 ('ก็', 18)
19 ('u', 19)


In [10]:
train_dataset = TokenNormDataset(
    data=train_tok,
    tokenizer=tokenizer,
    label2id=label2id,
    max_len=MAX_LEN,
)
val_dataset = TokenNormDataset(
    data=val_tok,
    tokenizer=tokenizer,
    label2id=label2id,
    max_len=MAX_LEN,
)

DataLoader 생성

In [11]:
train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    generator=g,
    pin_memory=True,)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    pin_memory=True,)

학습

In [12]:
model = train_model(
    train_loader=train_loader,
    val_loader=val_loader,
    model_name=MODEL_NAME,
    num_labels=num_labels,
    epochs=EPOCHS,
    lr=LR,
    #ckpt_path="token_norm_xlmr_checkpoint.pt",
    ckpt_path="token_norm_distilmbert_checkpoint.pt",
)
# 모델 단독 예측 평가
result_model_only = predict_with_mfr_fallback(
    model=model,
    tokenizer=tokenizer,
    data=val_tok,
    counts=counts,
    id2label=id2label,
    max_len=MAX_LEN,
    use_mfr_fallback=False,
    conf_threshold=0.0,
    mfr_ratio_threshold=1.1,
)

pred_model_only = result_model_only["pred"]

print("Model-only evaluation")
evaluate(
    raw_val,
    gold_val,
    pred_model_only,
    ignCaps=False,
    verbose=False,
    info=True,
)

device: cpu
🔥 checkpoint 로드: epoch 3부터 시작


Applying MFR fallback: 100%|██████████| 8408/8408 [00:00<00:00, 19328.69it/s]

Model-only evaluation
Baseline acc.(LAI): 88.48
Accuracy:           63.07
ERR:                -220.57


(0.8847954569019836, 0.6306898394442362, -2.2056909443367476)

MFR fallback 예측 평가
기본 규칙:
MFR ratio가 높으면 MFR 사용
아니고 model confidence가 threshold 이상이면 model 사용
그 외에는 MFR 사용

In [13]:
result_fallback = predict_with_mfr_fallback(
    model=model,
    tokenizer=tokenizer,
    data=val_tok,
    counts=counts,
    id2label=id2label,
    max_len=MAX_LEN,
    use_mfr_fallback=True,
    conf_threshold=0.80,
    mfr_ratio_threshold=0.95,
)

pred_fallback = result_fallback["pred"]

print("MFR + classifier fallback evaluation")
lai_fb, acc_fb, err_fb = evaluate(
    raw_val,
    gold_val,
    pred_fallback,
    ignCaps=False,
    verbose=False,
    info=True,
)

print("Fallback result")
print("LAI:", lai_fb)
print("Accuracy:", acc_fb)
print("ERR:", err_fb)

Applying MFR fallback: 100%|██████████| 8408/8408 [00:00<00:00, 15287.27it/s]

MFR + classifier fallback evaluation
Baseline acc.(LAI): 88.48
Accuracy:           93.45
ERR:                43.13
Fallback result
LAI: 0.8847954569019836
Accuracy: 0.934477615511617
ERR: 0.43125173082248747


threshold 튜닝

In [14]:
threshold_results = []

#for conf_th in [0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90, 0.95]:
for conf_th in [0.50, 0.95]:
    #for ratio_th in [0.80, 0.85, 0.90, 0.95, 0.98, 1.01]:
    for ratio_th in [0.80, 1.01]:
        result = predict_with_mfr_fallback(
            model=model,
            tokenizer=tokenizer,
            data=val_tok,
            counts=counts,
            id2label=id2label,
            max_len=MAX_LEN,
            use_mfr_fallback=True,
            conf_threshold=conf_th,
            mfr_ratio_threshold=ratio_th,
        )

        pred = result["pred"]

        lai, acc, err = evaluate(
            raw_val,
            gold_val,
            pred,
            ignCaps=False,
            verbose=False,
            info=False,
        )

        threshold_results.append({
            "conf_threshold": conf_th,
            "mfr_ratio_threshold": ratio_th,
            "lai": lai,
            "accuracy": acc,
            "err": err,
        })

threshold_df = pd.DataFrame(threshold_results)
threshold_df = threshold_df.sort_values("err", ascending=False)

threshold_df.head(20)

Applying MFR fallback: 100%|██████████| 8408/8408 [00:00<00:00, 17592.94it/s]


,conf_threshold,mfr_ratio_threshold,lai,accuracy,err
0,0.50,0.80,0.884795,0.933688,0.424398
3,0.95,1.01,0.884795,0.932739,0.416159
2,0.95,0.80,0.884795,0.932196,0.411451
1,0.50,1.01,0.884795,0.929548,0.388466


best threshold로 재평가

In [15]:
best = threshold_df.iloc[0]

best_conf_th = float(best["conf_threshold"])
best_ratio_th = float(best["mfr_ratio_threshold"])

print("best_conf_threshold:", best_conf_th)
print("best_mfr_ratio_threshold:", best_ratio_th)
print(best)

best_conf_threshold: 0.5
best_mfr_ratio_threshold: 0.8
conf_threshold         0.500000
mfr_ratio_threshold    0.800000
lai                    0.884795
accuracy               0.933688
err                    0.424398
Name: 0, dtype: float64


In [16]:
best_result = predict_with_mfr_fallback(
    model=model,
    tokenizer=tokenizer,
    data=val_tok,
    counts=counts,
    id2label=id2label,
    max_len=MAX_LEN,
    use_mfr_fallback=True,
    conf_threshold=best_conf_th,
    mfr_ratio_threshold=best_ratio_th,
)

best_pred = best_result["pred"]

evaluate(
    raw_val,
    gold_val,
    best_pred,
    ignCaps=False,
    verbose=False,
    info=True,
)

Applying MFR fallback: 100%|██████████| 8408/8408 [00:00<00:00, 18876.23it/s]

Baseline acc.(LAI): 88.48
Accuracy:           93.37
ERR:                42.44


(0.8847954569019836, 0.9336879970010449, 0.4243976737745781)

어떤 token이 MFR에서 모델로 바뀌었는지 확인

In [17]:
changes = []

for item, mfr_sent, pred_sent in zip(val_tok, mfr_pred_val, best_pred):
    raw_sent = item["raw"]
    gold_sent = item["norm"]

    for raw_tok, gold_tok, mfr_tok, pred_tok in zip(
        raw_sent,
        gold_sent,
        mfr_sent,
        pred_sent,
    ):
        if mfr_tok != pred_tok:
            changes.append({
                "raw": raw_tok,
                "gold": gold_tok,
                "mfr": mfr_tok,
                "pred": pred_tok,
                "mfr_correct": mfr_tok == gold_tok,
                "pred_correct": pred_tok == gold_tok,
            })

changes_df = pd.DataFrame(changes)

print("num changed from MFR:", len(changes_df))
changes_df.head(50)

num changed from MFR: 1288


,raw,gold,mfr,pred,mfr_correct,pred_correct
0,im,im,i'm,im,False,True
1,im,im,i'm,im,False,True
2,im,im,i'm,im,False,True
3,net,nicht,net,nicht,False,True
4,im,im,i'm,im,False,True
5,im,im,i'm,im,False,True
6,im,im,i'm,im,False,True
7,im,im,i'm,im,False,True
8,im,im,i'm,im,False,True
9,net,nicht,net,nicht,False,True


MFR보다 좋아진/나빠진 케이스 확인

In [18]:
if len(changes_df) > 0:
    print("Model fixed MFR errors:")
    display(changes_df[(changes_df["mfr_correct"] == False) & (changes_df["pred_correct"] == True)].head(50))

    print("Model broke MFR correct predictions:")
    display(changes_df[(changes_df["mfr_correct"] == True) & (changes_df["pred_correct"] == False)].head(50))
else:
    print("No differences from MFR.")

Model fixed MFR errors:


,raw,gold,mfr,pred,mfr_correct,pred_correct
0,im,im,i'm,im,False,True
1,im,im,i'm,im,False,True
2,im,im,i'm,im,False,True
3,net,nicht,net,nicht,False,True
4,im,im,i'm,im,False,True
5,im,im,i'm,im,False,True
6,im,im,i'm,im,False,True
7,im,im,i'm,im,False,True
8,im,im,i'm,im,False,True
9,net,nicht,net,nicht,False,True


Model broke MFR correct predictions:


,raw,gold,mfr,pred,mfr_correct,pred_correct
10,*g*,*g*,*g*,*,True,False
15,%NegSmileyen,%NegSmileyen,%NegSmileyen,%PosSmiley,True,False
17,[:,[:,[:,[,True,False
18,hahahaah,hahahaah,hahahaah,hahaha,True,False
20,^^;,^^;,^^;,^^,True,False
23,"""oil-painting""-Modus","""oil-painting""-Modus","""oil-painting""-Modus","""",True,False
26,%PosSmiley1,%PosSmiley1,%PosSmiley1,%PosSmiley,True,False
29,I,I,I,i,True,False
35,;;,;;,;;,;,True,False
36,imy,imy,imy,i'm,True,False


prediction 저장

In [19]:
output_df = pd.DataFrame({
    "raw": raw_val,
    "norm": gold_val,
    "mfr_pred": mfr_pred_val,
    "pred": best_pred,
})

output_df.to_pickle("val_predictions.pkl")
output_df.head()

,raw,norm,mfr_pred,pred
0,"[fänd, ich, toll, !, %PosSmiley]","[Fände, ich, toll, !, %PosSmiley]","[fänd, ich, toll, !, %PosSmiley]","[fänd, ich, toll, !, %PosSmiley]"
1,"[Deshalb, !]","[Deshalb, !]","[Deshalb, !]","[Deshalb, !]"
2,"[bei, einer, netten, tollen, Frau, liegen, !]","[Bei, einer, netten, tollen, Frau, liegen, !]","[bei, einer, netten, tollen, Frau, liegen, !]","[bei, einer, netten, tollen, Frau, liegen, !]"
3,"[warum, nicht, !]","[Warum, nicht, !]","[warum, nicht, !]","[warum, nicht, !]"
4,"[yay, ,, und, ich, hab, mir, heut, früh, n, ne...","[Yay, ,, und, ich, habe, mir, heute, früh, ein...","[yay, ,, und, ich, habe, mir, heute, früh, dan...","[yay, ,, und, ich, habe, mir, heute, früh, dan..."
